# Copycat

Copycat is a small concatenative language in which a model invocation is a
first-class program-synthesis effect. The deterministic runtime stays in
control: `{natural language}` asks a backend for Copycat code, and that code
continues immediately against the current data stack.

The language implementation now lives in the installable `copycat` package.
This notebook contains its interactive tests and examples.


## Setup — run this section first

Run these cells from the notebook directory to install Copycat in editable mode
and import its public API. The optional Gemma dependencies are installed only
in the live-model section below.


### Install the package


In [ ]:
%pip install -q "copycat-language[test] @ git+https://github.com/dearmadisonblue/copycat.git@main"


In [ ]:
from copycat import (
    CopycatError,
    EvaluationError,
    GeneratedCodeError,
    Gemma4Backend,
    Model,
    ModelProtocolError,
    ModelReportedError,
    ParseError,
    StubModel,
    read,
    run,
)


## Tests


In [ ]:
import ipytest
import pytest

ipytest.autoconfig()


In [ ]:
%%ipytest -q


@pytest.mark.parametrize(
    "source, expected",
    [
        ("[foo] Copy", "[foo] [foo]"),
        ("[foo] Drop", ""),
        ("[foo] [bar] Swap", "[bar] [foo]"),
        ("[foo] [bar] Cat", "[foo bar]"),
        ("[foo] Abs", "[[foo]]"),
        ("[foo] App", "foo"),
        ("[foo] Jump bar qux Mark baz", "[bar qux] foo baz"),
    ],
)
def test_original_examples(source, expected):
    assert run(source, verbose=False) == expected


def test_jump_finds_mark_when_mark_is_the_final_instruction():
    assert run("[App] Jump 1 Mark", verbose=False) == "1"


def test_model_form_is_opaque_to_copycat_syntax():
    program = read('{write [this] and "that"\non two lines}')
    (model,) = program.body
    assert isinstance(model, Model)
    assert model.prompt == 'write [this] and "that"\non two lines'


@pytest.mark.parametrize(
    "source, fragment",
    [
        ("[Copy", "Unclosed quotation"),
        ("Copy]", "no matching '['"),
        ("{do something", "Unclosed model invocation"),
        ('"unterminated', "unterminated string"),
        ("Copy @", "Unexpected character '@'"),
    ],
)
def test_parser_errors_are_explanatory(source, fragment):
    with pytest.raises(ParseError) as caught:
        read(source)
    assert fragment.lower() in str(caught.value).lower()


def test_strict_evaluation_reports_stack_underflow():
    with pytest.raises(EvaluationError) as caught:
        run("Copy", strict=True, verbose=False)

    message = str(caught.value)
    assert "Copy needs 1 value" in message
    assert "line 1, column 1" in message


def test_stub_model_ok_executes_generated_code_immediately():
    backend = StubModel("<OK>Swap</OK>")

    assert run(
        "1 2 {swap the top two values}",
        model_backend=backend,
        strict=True,
        verbose=False,
    ) == "2 1"

    assert backend.calls == [
        ("swap the top two values", "1 2")
    ]


def test_stub_model_error_becomes_structured_condition():
    backend = StubModel("<ERROR>I cannot do that safely.</ERROR>")

    with pytest.raises(ModelReportedError):
        run(
            "1 {do something impossible}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


def test_protocol_allows_surrounding_text_and_lowercase_tags():
    backend = StubModel("I chose this.\n<ok>Copy</ok>\nDone.")

    assert run(
        "1 {duplicate the value}",
        model_backend=backend,
        strict=True,
        verbose=False,
    ) == "1 1"


def test_protocol_rejects_multiple_expected_elements():
    backend = StubModel("<OK>Copy</OK> or <ERROR>unsure</ERROR>")

    with pytest.raises(ModelProtocolError):
        run(
            "1 {duplicate the value}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


def test_bad_generated_syntax_is_attributed_to_model_output():
    backend = StubModel("<OK>[Copy</OK>")

    with pytest.raises(GeneratedCodeError) as caught:
        run(
            "1 {do it}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )

    assert "<model output>" in str(caught.value)


def test_nested_model_calls_are_disabled_by_default():
    backend = StubModel("<OK>{ask again}</OK>")

    with pytest.raises(GeneratedCodeError):
        run(
            "1 {do it}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


## Examples


Run this section in Colab to exercise the reader, deterministic model effects,
then load Gemma and run the live examples. Evaluator tracing is intentionally
enabled in the executable examples.


### Reader examples


In [ ]:
for source in [
    "1 2 Swap",
    "[foo] Jump bar qux Mark baz",
    '{write [this] and "that" on two lines}',
]:
    print(f"{source!r} -> {read(source)!r}")


### Deterministic model examples


In [ ]:
swap_stub = StubModel("Reasoning outside the tag is accepted.\n<ok>Swap</ok>")
copy_stub = StubModel("<OK>Copy</OK>")

print(
    run(
        "1 2 {put the top two values in the opposite order}",
        model_backend=swap_stub,
        strict=True,
    )
)

print(
    run(
        '"hello" {duplicate the top value}',
        model_backend=copy_stub,
        strict=True,
    )
)


### Live Gemma 4 synthesis


#### Load Gemma


In [ ]:
%pip install -q -U "copycat-language[gemma] @ git+https://github.com/dearmadisonblue/copycat.git@main"


In [ ]:
# Optional, only if your Hugging Face environment asks for authentication:
# from huggingface_hub import notebook_login
# notebook_login()

gemma = Gemma4Backend.load(
    max_new_tokens=8_192,
    stream_output=True,
)


#### Run live examples


In [ ]:
examples = [
    "1 2 {put the top two values in the opposite order}",
    '"hello" {duplicate the top value}',
    "{put the number 7 on the stack}",
]

for source in examples:
    print("\n" + "=" * 72)
    print("SOURCE:", source)
    try:
        print(
            "RESULT:",
            run(
                source,
                model_backend=gemma,
                strict=True,
            ),
        )
    except CopycatError as exc:
        print(exc)


## Future work

This version remains deliberately narrow: one model turn synthesizes a small
Copycat program, that program is parsed, and ordinary evaluation continues.
Repair loops, generic effect handlers, capabilities, external services,
simulation, persisted continuations, actors, and nested model effects remain
deferred until this one-shot path has been exercised with the live checkpoint.
